# Shape-similarity visualization + prediction

**Date:** 2026-04-18.
**Jake's hypothesis:** maybe combined_score just isn't finding the right neighbors. If a movie with `the_drama`'s shape exists in the cohort but combined_score doesn't pick it, our similarity metric is wrong — not our cohort coverage.

**Procedure:**
1. Collapse all review timestamps to day-level (midnight UTC of their day) — makes h/m and day-level directly comparable.
2. Plot per-movie cumulative arrival curves, aligned at first-review-day = 0. Highlight h/m targets.
3. Compute pairwise shape-similarity via cosine similarity of daily arrival vectors.
4. For each h/m target, find top-20 most shape-similar candidates. Compare to combined_score's top-20.
5. Build weighted KDE from shape-similar training. Predict phase-1 (midnight+noon convention). Compare MAE to combined_score-based prediction.

**If shape-similar ≠ combined_score AND produces better predictions** → similarity is miscalibrated, add shape as a feature.

**If shape-similar ≈ combined_score** → cohort coverage confirmed, accept it.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP_DAYS = 3

HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']

CACHE = CACHE_DIR / 'shape_viz.pkl'

# Collapse all reviews to day-level; also noon-shift day-level ones
reviews_day = reviews.copy()
reviews_day['estimated_timestamp'] = reviews_day['estimated_timestamp'].dt.floor('D')
# For noon-shifted training analogue, add 12h (so both h/m and day-level sit at noon)
reviews_noon = reviews.copy()
day_mask = reviews_noon['timestamp_confidence'] == 'd'
reviews_noon.loc[day_mask, 'estimated_timestamp'] = (
    reviews_noon.loc[day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)

print('Frames:')
print(f'  reviews (orig): {len(reviews):,} rows')
print(f'  reviews_day (all floored to day): {len(reviews_day):,}')
print(f'  reviews_noon (day-level shifted to noon): {len(reviews_noon):,}')

## Daily arrival vectors per movie

For each resolved movie, compute `daily_count[d]` where `d = 0` is the day of the first review and `d = N` is N days later. Limit to the pre-close period.

In [ ]:
def daily_arrivals(slug, reviews_df, max_days=None):
    """Daily arrival counts since first review day, up to the movie's close day."""
    close_ts = close_date_map[slug]
    mr = reviews_df[reviews_df['movie_slug'] == slug].copy()
    if len(mr) == 0:
        return None
    mr = mr.sort_values('estimated_timestamp')
    # Day index since first review (0-indexed)
    first_day = mr['estimated_timestamp'].iloc[0].floor('D')
    mr['day_idx'] = ((mr['estimated_timestamp'].dt.floor('D') - first_day) / pd.Timedelta(days=1)).astype(int)
    # Cap at close day - 1 (so we drop close-day reviews for consistency)
    close_day = close_ts.floor('D')
    close_day_idx = int((close_day - first_day) / pd.Timedelta(days=1))
    mr = mr[mr['day_idx'] < close_day_idx]
    daily = mr.groupby('day_idx').size()
    # Build full array [0, close_day_idx - 1]
    arr = np.zeros(close_day_idx, dtype=int)
    for idx, count in daily.items():
        if 0 <= idx < close_day_idx:
            arr[idx] = count
    if max_days is not None:
        arr = arr[:max_days]
    return arr

# Build daily vectors for all movies (using day-collapsed reviews)
daily_vecs = {}
for slug in close_date_map:
    v = daily_arrivals(slug, reviews_day, max_days=30)
    if v is not None and v.sum() > 0:
        daily_vecs[slug] = v

print(f'Built daily vectors for {len(daily_vecs)} movies (capped at 30 days post first review)')
print(f'Vector length distribution: {pd.Series([len(v) for v in daily_vecs.values()]).describe().round(1).to_dict()}')

## Visualization: cumulative arrival curves, aligned at first-review-day

All movies plotted faintly, h/m highlighted. Look for movies whose curves share shape with h/m targets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for slug, vec in daily_vecs.items():
    if slug in HM:
        continue
    cum = np.cumsum(vec)
    axes[0].plot(cum, color='lightgray', alpha=0.4, linewidth=0.6)
    axes[1].plot(vec, color='lightgray', alpha=0.4, linewidth=0.6)

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
for i, slug in enumerate(HM):
    if slug not in daily_vecs:
        continue
    vec = daily_vecs[slug]
    cum = np.cumsum(vec)
    axes[0].plot(cum, color=colors[i], linewidth=2.5, label=slug)
    axes[1].plot(vec, color=colors[i], linewidth=2.5, label=slug)

axes[0].set_xlabel('Days since first review')
axes[0].set_ylabel('Cumulative review count')
axes[0].set_title('Cumulative arrival curves (day-level collapsed)')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].set_xlim(0, 30)

axes[1].set_xlabel('Days since first review')
axes[1].set_ylabel('Reviews on this day')
axes[1].set_title('Daily arrival rate (day-level collapsed)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_xlim(0, 30)

plt.tight_layout()
out_path = ROOT.parent / 'notebooks' / 'shape_viz_curves.png' if ROOT.name == 'notebooks' else ROOT / 'notebooks' / 'shape_viz_curves.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {out_path}')


## Shape-similarity matrix

For each movie pair, compare daily arrival vectors via cosine similarity. Uses vectors truncated to common length (shortest wins).

In [ ]:
def cosine_sim(a, b):
    n = min(len(a), len(b))
    if n == 0:
        return 0.0
    a, b = a[:n], b[:n]
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

slugs_ordered = list(daily_vecs.keys())
n = len(slugs_ordered)
sim_matrix = np.zeros((n, n))
for i, s1 in enumerate(slugs_ordered):
    for j, s2 in enumerate(slugs_ordered):
        if i == j:
            sim_matrix[i,j] = 1.0
        else:
            sim_matrix[i,j] = cosine_sim(daily_vecs[s1], daily_vecs[s2])

print(f'Similarity matrix: {sim_matrix.shape}')
sim_df = pd.DataFrame(sim_matrix, index=slugs_ordered, columns=slugs_ordered)

# For each h/m target, show top 20 shape-similar candidates
for target in HM:
    if target not in slugs_ordered:
        continue
    sims = sim_df[target].drop(target)
    # Filter to movies with close date BEFORE target's close (for fair training selection)
    target_close = close_date_map[target]
    past_slugs = [s for s in sims.index if close_date_map[s] < target_close]
    sims = sims[past_slugs]
    top = sims.nlargest(10)
    print(f'\n{target} — top 10 shape-similar past movies:')
    for cand, s in top.items():
        gap_cand = gap_for_slug(cand)
        print(f'  {cand:45s}  cos_sim={s:.3f}  gap={gap_cand:.1f}d')

## Compare shape-similar vs combined_score picks

In [ ]:
def get_combined_score_top_k(target, snap_days=SNAP_DAYS, k=20):
    target_gap = gap_for_slug(target)
    target_close = close_date_map[target]
    snap_time = target_close.floor('D') - pd.Timedelta(days=snap_days)
    mr = reviews[reviews['movie_slug'] == target]
    obs = mr[mr['estimated_timestamp'] < snap_time]
    if len(obs) == 0:
        return {}
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
    }
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
    target_window_days = state['first_review_dbc'] - snap_dbc_eff
    if target_window_days <= 0:
        return {}
    return combined_score_with_scores(
        target, target_gap, state['observed_critics'], target_window_days,
        k=k, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )

print('For each h/m target, overlap between shape-similar top-20 and combined_score top-20:\n')
for target in HM:
    if target not in slugs_ordered:
        continue
    target_close = close_date_map[target]
    past_slugs = [s for s in slugs_ordered if close_date_map[s] < target_close and s != target]
    shape_top = sim_df[target].loc[past_slugs].nlargest(20).index.tolist()
    cs_top = list(get_combined_score_top_k(target, k=20).keys())
    overlap = set(shape_top) & set(cs_top)
    print(f'{target}:')
    print(f'  shape top-20 (first 10): {shape_top[:10]}')
    print(f'  cs top-20 (first 10):    {cs_top[:10]}')
    print(f'  overlap size: {len(overlap)}/20')
    print()

## Prediction test: does shape-similar training improve h/m MAE?

Build weighted KDE from shape-similar top-20 (weights = cosine similarity). Predict phase-1 under midnight+noon convention. Compare to weighted KDE from combined_score top-20.

In [ ]:
def predict_with_training_scores(target, training_scores_dict, snap_days=SNAP_DAYS):
    """Build weighted KDE from the given scored training set, predict phase_1."""
    target_close = close_date_map[target]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=snap_days)  # midnight-aligned
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400

    # Compute observed state on noon-shifted reviews (for midnight+noon convention)
    obs = reviews_noon[
        (reviews_noon['movie_slug'] == target)
        & (reviews_noon['estimated_timestamp'] < snap_time)
        & (reviews_noon['estimated_timestamp'] < target_close)
    ]
    if len(obs) == 0:
        return None
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
    }
    if state['first_review_dbc'] < snap_dbc_eff + 1.0:
        return None
    # Build weighted profiles using noon-shifted training
    training_slugs = list(training_scores_dict.keys())
    n_movies = len(training_slugs)
    raw_weights = np.array([training_scores_dict[s] for s in training_slugs], dtype=float)
    total_w = raw_weights.sum()
    if total_w <= 0:
        norm_weights = np.ones_like(raw_weights)
    else:
        norm_weights = raw_weights * (n_movies / total_w)
    slug_weight = dict(zip(training_slugs, norm_weights))
    train = reviews_noon[reviews_noon['movie_slug'].isin(training_slugs)].copy()
    close_map_series = pd.Series(close_date_map)
    train['bet_close'] = train['movie_slug'].map(close_map_series)
    train['days_before_close'] = (train['bet_close'] - train['estimated_timestamp']).dt.total_seconds() / 86400
    train = train[train['days_before_close'] > 0].copy()
    train['movie_weight'] = train['movie_slug'].map(slug_weight)
    rows = []
    for name, group in train.groupby('reviewer_name'):
        movies_seen = group['movie_slug'].unique()
        base_rate = float(sum(slug_weight[s] for s in movies_seen) / n_movies)
        fresh = (group['tomatometer_sentiment'] == 'positive').sum()
        total = len(group)
        timing = group['days_before_close'].values.tolist()
        weights_list = group['movie_weight'].values.tolist()
        rows.append({'reviewer_name': name, 'base_rate': base_rate,
                     'fresh_rate': fresh / total if total > 0 else 0.5,
                     'timing_data': timing, 'timing_weights': weights_list, 'n_reviews': total})
    df = pd.DataFrame(rows, columns=['reviewer_name','base_rate','fresh_rate','timing_data','timing_weights','n_reviews'])
    if len(df) == 0:
        return None
    from rotten_tomatoes_forecasting.critic_model import CriticProfiles
    profiles = CriticProfiles(df=df, training_slug_count=n_movies)
    model = build_weighted_kde_lambda_model(
        profiles, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
    )
    pred = predict_window_custom(
        model, dbc_from=snap_dbc_eff, dbc_to=midnight_utc_dbc,
        observed_critics=state['observed_critics'],
        observed_count=state['observed_count'],
        first_review_dbc=state['first_review_dbc'],
    )
    # Compute actual under midnight+noon convention
    close_midnight = target_close.floor('D')
    mr_all = reviews_noon[reviews_noon['movie_slug'] == target]
    actual_p1 = int(((mr_all['estimated_timestamp'] >= snap_time) & (mr_all['estimated_timestamp'] < close_midnight)).sum())
    return float(pred), actual_p1


print('Prediction comparison: combined_score vs shape-similar training:\n')
print(f'  {"target":32s} {"cs_pred":>8s} {"shape_pred":>11s} {"actual":>7s} {"cs_err":>8s} {"shape_err":>10s}')
for target in HM:
    if target not in slugs_ordered:
        continue
    target_close = close_date_map[target]
    past_slugs = [s for s in slugs_ordered if close_date_map[s] < target_close and s != target]
    shape_sims = sim_df[target].loc[past_slugs].nlargest(20)
    shape_scores = dict(shape_sims)

    cs_scores = get_combined_score_top_k(target, k=20)
    if not cs_scores:
        continue

    r_cs = predict_with_training_scores(target, cs_scores)
    r_shape = predict_with_training_scores(target, shape_scores)
    if r_cs is None or r_shape is None:
        continue
    cs_pred, actual = r_cs
    shape_pred, actual2 = r_shape
    assert actual == actual2
    cs_err = cs_pred - actual
    shape_err = shape_pred - actual
    print(f'  {target:32s} {cs_pred:8.2f} {shape_pred:11.2f} {actual:7d} {cs_err:+8.2f} {shape_err:+10.2f}')

## Zoom in on the_drama

Plot the_drama's daily arrival curve alongside its top-20 shape-similar AND top-20 combined_score training sets. Visual intuition.

In [ ]:
target = 'the_drama'
target_close = close_date_map[target]
past_slugs = [s for s in slugs_ordered if close_date_map[s] < target_close and s != target]
shape_top = sim_df[target].loc[past_slugs].nlargest(20).index.tolist()
cs_top = list(get_combined_score_top_k(target, k=20).keys())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for slug in shape_top:
    if slug in daily_vecs:
        axes[0].plot(daily_vecs[slug], color='lightgray', alpha=0.5, linewidth=1)
if target in daily_vecs:
    axes[0].plot(daily_vecs[target], color='red', linewidth=2.5, label=target)
axes[0].set_title(f'{target} vs top-20 shape-similar (daily arrivals)')
axes[0].set_xlabel('Days since first review')
axes[0].set_ylabel('Reviews on this day')
axes[0].legend()
axes[0].set_xlim(0, 30)

for slug in cs_top:
    if slug in daily_vecs:
        axes[1].plot(daily_vecs[slug], color='lightgray', alpha=0.5, linewidth=1)
if target in daily_vecs:
    axes[1].plot(daily_vecs[target], color='red', linewidth=2.5, label=target)
axes[1].set_title(f'{target} vs top-20 combined_score (daily arrivals)')
axes[1].set_xlabel('Days since first review')
axes[1].set_ylabel('Reviews on this day')
axes[1].legend()
axes[1].set_xlim(0, 30)

plt.tight_layout()
out_path = ROOT.parent / 'notebooks' / 'shape_viz_the_drama.png' if ROOT.name == 'notebooks' else ROOT / 'notebooks' / 'shape_viz_the_drama.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {out_path}')


## Interpretation

- If shape-similar training gives predictions meaningfully closer to actual for h/m targets, similarity formula was miscalibrated — add shape as a feature.
- If predictions are nearly identical, combined_score was already picking shape-reasonable neighbors (just not "perfectly" shape-matched, but close enough).
- If shape-similar predictions are WORSE, cosine-on-rate isn't capturing the right similarity dimension.

The left-panel plots visualize whether the h/m curves have obvious cohort neighbors at all.